In [1]:
import numpy as np
import pandas as pd
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

TEAL, INK, GREY, MID, LIGHT = "#36B7B4", "#122b39", "#676A86", "#7A828E", "#B8BFC9"

# IMF World Economic Outlook, October 2026, Figure 1.8 (panel 4), 2025 data.
# Printed as a chart, not a table: values read off the figure's vector geometry (about +/-0.1).
# years = average years to maturity of outstanding debt; bills = Treasury bills as a share of
# gross borrowing (%); refi = refinancing requirement, % of GDP (the labels printed in the WEO).
DEBT = pd.DataFrame([
    ("United States",  5.9, 58, 30.8),
    ("Italy",          6.9, 27, 16.8),
    ("Germany",        7.6, 27,  5.5),
    ("France",         8.5, 37, 14.2),
    ("Japan",          9.4, 24, 24.6),
    ("United Kingdom", 13.5, 21,  8.7),
], columns=["country", "years", "bills", "refi"])

HIGHLIGHT = {"United States": INK, "United Kingdom": TEAL}
d = DEBT.copy()
d["lcol"] = d.country.map(lambda c: HIGHLIGHT.get(c, MID))      # label colour
d["dot"] = d.country.map(lambda c: HIGHLIGHT.get(c, LIGHT))     # bubble colour
d["lab"] = d.country + "  " + d.refi.map(lambda v: f"{v:.1f}")
MAX_AREA = 2600
d["r_px"] = np.sqrt(d.refi / 31 * MAX_AREA / np.pi)             # bubble radius in px (area ~ refinancing need)

BX = alt.Scale(domain=[4, 16], nice=False)
BY = alt.Scale(domain=[0, 70], nice=False)
bxa = alt.Axis(title="Average years to maturity of outstanding government debt", grid=False, values=[4, 6, 8, 10, 12, 14, 16])
bya = alt.Axis(title="Treasury bills as a share of gross borrowing (%)", values=[0, 20, 40, 60])

bubbles = alt.Chart(d).mark_circle(opacity=1, stroke="white", strokeWidth=1.5).encode(
    x=alt.X("years:Q", scale=BX, axis=bxa), y=alt.Y("bills:Q", scale=BY, axis=bya),
    size=alt.Size("refi:Q", scale=alt.Scale(domain=[0, 31], range=[0, MAX_AREA]), legend=None),
    color=alt.Color("dot:N", scale=None, legend=None))

# ---- labels: offsets in pixels from each bubble's centre (dx right, dy down) ----------------
# Crowded bubbles (Germany, France) get a leader line to a label in clear space; the rest sit beside
# their bubble. Tweak the numbers below to move a label; everything else follows.
PX_PER_YEAR, PX_PER_PCT = 600 / 12, 330 / 70          # chart width/height over the axis ranges
LABELS = {  # country: (dx, dy, align, leader?)
    "United States":  ( 52,   0, "left",   False),
    "Japan":          ( 44,   2, "left",   False),
    "United Kingdom": (  0, -34, "center", False),
    "Italy":          (-38,   0, "right",  False),
    "Germany":        ( 16,  62, "left",   True),
    "France":         ( 32, -58, "left",   True),
}
lab = d.copy()
lab["dx"] = lab.country.map(lambda c: LABELS[c][0])
lab["dy"] = lab.country.map(lambda c: LABELS[c][1])
lab["align"] = lab.country.map(lambda c: LABELS[c][2])
lab["leader"] = lab.country.map(lambda c: LABELS[c][3])
lab["lx"] = lab.years + lab.dx / PX_PER_YEAR            # label anchor, data units
lab["ly"] = lab.bills - lab.dy / PX_PER_PCT

# leader runs from the bubble's centre to just short of the label; it is drawn UNDER the bubbles
# (opaque), so it appears to meet the edge exactly, whatever the bubble size
length = np.hypot(lab.dx, lab.dy).replace(0, 1)
ux, uy = lab.dx / length, lab.dy / length
lab["x1"], lab["y1"] = lab.years, lab.bills
lab["x2"] = lab.years + (lab.dx - ux * 5) / PX_PER_YEAR
lab["y2"] = lab.bills - (lab.dy - uy * 5) / PX_PER_PCT

leaders = alt.Chart(lab[lab["leader"].astype(bool)]).mark_rule(color=MID, strokeWidth=1, opacity=1).encode(
    x=alt.X("x1:Q", scale=BX), y=alt.Y("y1:Q", scale=BY), x2="x2:Q", y2="y2:Q")

labels = [alt.Chart(lab[lab["align"] == al]).mark_text(
              align=al, baseline="middle", dx=0, dy=0, fontSize=12, fontWeight="bold", opacity=1).encode(
              x=alt.X("lx:Q", scale=BX), y=alt.Y("ly:Q", scale=BY), text="lab:N",
              color=alt.Color("lcol:N", scale=None, legend=None))
          for al in ("left", "right", "center")]

chart = alt.layer(leaders, bubbles, *labels).properties(width=400, height=300)

chart = chart.configure(background="white", font="Circular Std").configure_view(
    fill="transparent", stroke="transparent")
styles.save(chart, path=".", name="weo_refinancing", svg=True)
chart.save("weo_refinancing.png", scale_factor=2.0)
chart

alt.LayerChart(...)